In [1]:
import numpy as np

<a id="weighted-sampling"></a>
# 1. Weighted sampling — highest priority

### Practice problem

Given `items` and nonnegative `weights` of equal length, draw items **with replacement** so that


$$
P(\text{item at index }i)=\frac{w_i}{\sum_j w_j}.
$$

| Input / output | Contract |
|---|---|
| `items` | Nonempty sequence. We sample **positions**; repeated labels are allowed and their probabilities add. |
| `weights` | Ordinary Python `int` or `float` values, nonnegative and finite; at least one is positive. They need not sum to 1. |
| `k` | Number of independent draws; `0` returns an empty list for a valid distribution. |
| `rng` | A random-number generator object. Supplying it makes examples reproducible and tests controllable. |
| Result | The function returns a **list of items**, even when `k=1`. The class also exposes a one-item `sample()` method. |

**Ask before coding:** Are weights integers or floats? Static or updated? Replacement or no replacement? Return IDs or indexes? One draw or many?

We support integer and floating-point weights with the same cumulative-weight representation. Integer weights use an integer draw; float weights use a real-valued draw. [R1]

### Intuition: each weight owns an interval

```text
items:       A   B   C   D
weights:     1   3   0   2
cumulative:  1   4   4   6
```

| Item | Interval | Integer tickets in it | Probability |
|---|---|---|---|
| A | `[0, 1)` | `0` | `1/6` |
| B | `[1, 4)` | `1, 2, 3` | `3/6` |
| C | `[4, 4)` | None | `0` |
| D | `[4, 6)` | `4, 5` | `2/6` |

**Algorithm:** draw a ticket in `[0, total)` and find the first cumulative value **strictly greater** than it.

A ticket of `4` belongs to **D**. `bisect_right([1, 4, 4, 6], 4)` returns index `3`, skipping both cumulative `4`s. `bisect_left` would give the wrong bucket for this interval convention. [R2]

**Why it works:** the interval for position `i` has length `w_i`. Uniform sampling across total length `sum(weights)` allocates the required probability to that position. For integer weights, this is exactly a count of equally likely tickets.

In [15]:
def euclidean_similarity(vec1, vec2):

# Euclidean similarity is the inverse of distance (1 / (1 + distance))

    distance = np.linalg.norm(np.array(vec1) - np.array(vec2)[None,:])

    return 1 / (1 + distance)

In [4]:
existing_customers = [

[1, 0, 1, 0, 1], # User 1

[0, 1, 0, 1, 0], # User 2

[1, 1, 1, 0, 0], # User 3

[0, 0, 1, 1, 1], # User 4

[1, 0, 0, 1, 0], # User 5

[0, 1, 0, 0, 1], # User 6

[1, 1, 0, 1, 1], # User 7

[0, 0, 1, 1, 0], # User 8

[1, 1, 1, 1, 0], # User 9

[0, 1, 1, 0, 1], # User 10

]

target = [0, 0, 1, 0, 0]

In [5]:
euclidean_similarity([1, 0, 1, 0, 1],[0, 0, 1, 0, 0])

0.4142135623730951

In [16]:
euclidean_similarity(existing_customers, target)

0.1639607805437114

In [13]:
np.array(existing_customers) - np.array(target)[None,:]

array([[ 1,  0,  0,  0,  1],
       [ 0,  1, -1,  1,  0],
       [ 1,  1,  0,  0,  0],
       [ 0,  0,  0,  1,  1],
       [ 1,  0, -1,  1,  0],
       [ 0,  1, -1,  0,  1],
       [ 1,  1, -1,  1,  1],
       [ 0,  0,  0,  1,  0],
       [ 1,  1,  0,  1,  0],
       [ 0,  1,  0,  0,  1]])

In [21]:
1/(1+ np.linalg.norm(np.array(existing_customers)-np.array(target)[None,:], axis = 1))

array([0.41421356, 0.3660254 , 0.41421356, 0.41421356, 0.3660254 ,
       0.3660254 , 0.30901699, 0.5       , 0.3660254 , 0.41421356])